#### Read in avoided EAD files which show where the infrastructure damages are concentrated

In [ ]:
from pathlib import Path
import rasterio
from rasterio.plot import plotting_extent
import numpy as np
import matplotlib as mpl
from matplotlib import font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, NullLocator, ScalarFormatter
import geopandas as gpd
# from jamaica import plot
import sys, pathlib
sys.path.append(str(pathlib.Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers/robyns_libraries")))
import Robyn_paper_2_defs
import Robyn_river_floods
from matplotlib.colors import PowerNorm


In [ ]:
mpl.rcParams.update(Robyn_paper_2_defs.NATURE_RC)
print()

J2USD = 1.0 / 150.0  # conversion factor from Jamaican dollars to US dollars


In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
output_dir = base_path / "dphil_paper_2/results"
out_dir = output_dir / "figures"

In [ ]:
avoided_fluvial_ead_min_3448 = base_path / "dphil_paper_2/processed_data/nbs_river_catchment/avoided_fluvial_ead_min_3448.tif"
avoided_fluvial_ead_min_300m_smoothed = base_path / "dphil_paper_2/processed_data/nbs_river_catchment/avoided_fluvial_eads/avoided__fluvial__ead_min_300m_smoothed.tif"

In [ ]:
jamaica_boundary_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(jamaica_boundary.crs)

In [ ]:
with rasterio.open(avoided_fluvial_ead_min_300m_smoothed) as src:
    A = src.read(1, masked=True)

reproj = avoided_fluvial_ead_min_300m_smoothed

# Read raster & extent
with rasterio.open(reproj) as src:
    A = src.read(1, masked=True)

A.sum() * 1e-9

In [ ]:
# === Fig. 3(a/b) — Avoided EADs raster with colorbar in plain USD ============
# # data JD → thousands of USD; mask nonpositives
# data_kusd = np.A(arr, dtype="float64") * J2USD * 1e-3
# data_kusd[arr.mask | (data_kusd <= 0)] = np.nan

reproj = globals().get("avoided_fluvial_ead_min_300m_smoothed")
if reproj is None:
    raise ValueError("Set `avoided_fluvial_ead_min_300m_smoothed` to your raster path.")

# NATURE_RC = globals().get("NATURE_RC", {})

with rasterio.open(reproj) as src:
    A = src.read(1, masked=True)  # masked array; respects NoData
    try:
        plotting_extent_fn = globals().get("plotting_extent")
        if callable(plotting_extent_fn):
            extent = plotting_extent_fn(src)
        else:
            b = src.bounds
            extent = (b.left, b.right, b.bottom, b.top)
    except Exception:
        b = src.bounds
        extent = (b.left, b.right, b.bottom, b.top)
    r_crs = src.crs

# ---------- convert to USD; mask tiny/negative --------------------------------
# Keep your “≤ 1 J$” noise threshold, expressed in USD:
usd_noise_thresh = 1.0 * J2USD

data_usd = np.array(A, dtype="float64") * J2USD
data_usd[A.mask | (data_usd <= usd_noise_thresh)] = np.nan

# Positive-only stats for robust vmax
pos = data_usd[~np.isnan(data_usd)]
if pos.size == 0:
    raise ValueError("No positive values to plot (all zeros/NaNs).")
hi_usd = float(np.percentile(pos, 99.5))
if not np.isfinite(hi_usd) or hi_usd <= 0:
    hi_usd = float(np.nanmax(pos)) if np.isfinite(np.nanmax(pos)) else 1.0

# ---------- visualization params ---------------------------------------------
norm = mpl.colors.Normalize(vmin=0.0, vmax=hi_usd)  # LINEAR in USD
cmap = mpl.colormaps["magma_r"].copy()
cmap.set_bad((0, 0, 0, 0))  # NoData transparent

with mpl.rc_context(Robyn_paper_2_defs.NATURE_RC):
    TITLE_FS = mpl.rcParams.get("figure.titlesize", 7)
    LABEL_FS = mpl.rcParams.get("axes.labelsize", 6)
    TICK_FS  = mpl.rcParams.get("ytick.labelsize", mpl.rcParams.get("xtick.labelsize", 5.5))

    fig, ax = plt.subplots(figsize=(Robyn_river_floods.mm_to_in(90), Robyn_river_floods.mm_to_in(60)))
    ax.set_axis_off()

    # Raster (already in USD)
    im = ax.imshow(
        data_usd, norm=norm, cmap=cmap, extent=extent,
        origin="upper", interpolation="nearest"
    )

    # Jamaica outline (white casing + black), reprojected to raster CRS
    outline_gdf = None
    try:
        jamaica_boundary = globals()["jamaica_boundary"]
        outline = jamaica_boundary.union_all() if hasattr(jamaica_boundary, "union_all") else jamaica_boundary.unary_union
        outline_gdf = gpd.GeoSeries([outline], crs=getattr(jamaica_boundary, "crs", None))
        if getattr(outline_gdf, "crs", None) is not None and r_crs is not None:
            outline_gdf = outline_gdf.to_crs(r_crs)
        outline_gdf.plot(ax=ax, facecolor="none", edgecolor="white", linewidth=1.0, zorder=5)
        outline_gdf.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.45, zorder=6)
    except Exception as e:
        print("Outline skipped:", e)
        outline_gdf = None

    # Colorbar in **plain USD**
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.024, pad=0.012)
    cbar.ax.yaxis.set_minor_locator(NullLocator())
    cbar.outline.set_linewidth(0.35)

    locator = MaxNLocator(nbins=6, steps=[1, 2, 2.5, 5, 10], min_n_ticks=4)
    ticks_usd = locator.tick_values(0, hi_usd)
    ticks_usd = ticks_usd[(ticks_usd >= 0) & (ticks_usd <= hi_usd + 1e-12)]
    cbar.set_ticks(ticks_usd)

    def _fmt_usd(y):
        if y < 10:
            return f"{y:.2f}"
        elif y < 100:
            return f"{y:.1f}"
        else:
            return f"{y:,.0f}"  # thousands separators
    cbar.set_ticklabels([_fmt_usd(t) for t in ticks_usd])
    cbar.set_label("Avoided EADs (US$)", fontsize=LABEL_FS)
    cbar.ax.tick_params(labelsize=TICK_FS, width=0.35, length=2)

    # Scale bar & north arrow
    if outline_gdf is not None and r_crs and r_crs.is_projected:
        pt = Robyn_paper_2_defs.add_scale_bar(
            ax, outline_gdf, where="right-top",
            pad=0.07, length_km="auto", max_frac=0.22,
            lw=0.5, tick_h_frac=0.012, fs_lab=5, fs_unit=5, unit_text="km"
        )
        if pt is not None:
            cx_data, cy_data = pt
            cx_ax, cy_ax = ax.transAxes.inverted().transform(ax.transData.transform((cx_data, cy_data)))
            Robyn_paper_2_defs.add_north_arrow_axes(
                ax, cx_ax, cy_ax,
                size_frac=0.080, gap_frac=0.050,
                shaft_w_frac=0.10, head_w_frac=0.30, head_h_frac=0.50,
                fs=5, lw=0.5
            )
    # Save
    out_png = out_dir / "avoided_fluvial_ead_300m_linear_USD_plain_min_supplementary.png"
    fig.savefig(out_png, dpi=600, bbox_inches="tight", facecolor="white")
    plt.show()
    print("Saved:", out_png)

In [ ]:
# === Fig. 3(a/b) — Avoided EADs raster with colorbar in plain USD ============
# # data JD → thousands of USD; mask nonpositives
# data_kusd = np.A(arr, dtype="float64") * J2USD * 1e-3
# data_kusd[arr.mask | (data_kusd <= 0)] = np.nan

vmax = 25_000  # desired colorbar top in USD
gamma = 0.4
norm = mpl.colors.Normalize(vmin=0, vmax=25_000)



reproj = globals().get("avoided_fluvial_ead_min_300m_smoothed")
if reproj is None:
    raise ValueError("Set `avoided_fluvial_ead_min_300m_smoothed` to your raster path.")

with rasterio.open(reproj) as src:
    A = src.read(1, masked=True)  # masked array; respects NoData
    try:
        plotting_extent_fn = globals().get("plotting_extent")
        if callable(plotting_extent_fn):
            extent = plotting_extent_fn(src)
        else:
            b = src.bounds
            extent = (b.left, b.right, b.bottom, b.top)
    except Exception:
        b = src.bounds
        extent = (b.left, b.right, b.bottom, b.top)
    r_crs = src.crs

# ---------- convert to USD; mask tiny/negative --------------------------------
# Keep your “≤ 1 J$” noise threshold, expressed in USD:
usd_noise_thresh = 1.0 * J2USD

data_usd = np.array(A, dtype="float64") * J2USD
data_usd[A.mask | (data_usd <= usd_noise_thresh)] = np.nan

# Positive-only stats for robust vmax
pos = data_usd[~np.isnan(data_usd)]
if pos.size == 0:
    raise ValueError("No positive values to plot (all zeros/NaNs).")
hi_usd = float(np.percentile(pos, 99.5))
if not np.isfinite(hi_usd) or hi_usd <= 0:
    hi_usd = float(np.nanmax(pos)) if np.isfinite(np.nanmax(pos)) else 1.0

# ---------- visualization params ---------------------------------------------
# norm = mpl.colors.PowerNorm(vmin=0.0, vmax=hi_usd)  # LINEAR in USD
cmap = mpl.colormaps["magma_r"].copy()
cmap.set_bad((0, 0, 0, 0))  # NoData transparent

with mpl.rc_context(Robyn_paper_2_defs.NATURE_RC):
    TITLE_FS = mpl.rcParams.get("figure.titlesize", 7)
    LABEL_FS = mpl.rcParams.get("axes.labelsize", 6)
    TICK_FS  = mpl.rcParams.get("ytick.labelsize", mpl.rcParams.get("xtick.labelsize", 5.5))

    fig, ax = plt.subplots(figsize=(Robyn_river_floods.mm_to_in(90), Robyn_river_floods.mm_to_in(60)))
    ax.set_axis_off()

    # Raster (already in USD)
    im = ax.imshow(
        data_usd, norm=norm, cmap=cmap, extent=extent,
        origin="upper", interpolation="nearest"
    )

    # Jamaica outline (white casing + black), reprojected to raster CRS
    outline_gdf = None
    try:
        jamaica_boundary = globals()["jamaica_boundary"]
        outline = jamaica_boundary.union_all() if hasattr(jamaica_boundary, "union_all") else jamaica_boundary.unary_union
        outline_gdf = gpd.GeoSeries([outline], crs=getattr(jamaica_boundary, "crs", None))
        if getattr(outline_gdf, "crs", None) is not None and r_crs is not None:
            outline_gdf = outline_gdf.to_crs(r_crs)
        outline_gdf.plot(ax=ax, facecolor="none", edgecolor="white", linewidth=1.0, zorder=5)
        outline_gdf.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.45, zorder=6)
    except Exception as e:
        print("Outline skipped:", e)
        outline_gdf = None

    # Colorbar in **plain USD**
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.024, pad=0.012)
    cbar.ax.yaxis.set_minor_locator(NullLocator())
    cbar.outline.set_linewidth(0.35)

    locator = MaxNLocator(nbins=6, steps=[1, 2, 2.5, 5, 10], min_n_ticks=4)
    # build ticks from the same vmax
    ticks_usd = locator.tick_values(0, 25_000)
    ticks_usd = ticks_usd[(ticks_usd >= 0) & (ticks_usd <= 25_000 + 1e-12)]
    cbar.set_ticks(ticks_usd)
    cbar.set_ticklabels([_fmt_usd(t) for t in ticks_usd])



    def _fmt_usd(y):
        if y < 10:
            return f"{y:.2f}"
        elif y < 100:
            return f"{y:.1f}"
        else:
            return f"{y:,.0f}"  # thousands separators
    cbar.set_ticklabels([_fmt_usd(t) for t in ticks_usd])
    cbar.set_label("Avoided EADs (US$)", fontsize=LABEL_FS)
    cbar.ax.tick_params(labelsize=TICK_FS, width=0.35, length=2)

    # Scale bar & north arrow
    if outline_gdf is not None and r_crs and r_crs.is_projected:
        pt = Robyn_paper_2_defs.add_scale_bar(
            ax, outline_gdf, where="right-top",
            pad=0.07, length_km=20, max_frac=0.22,
            lw=0.5, tick_h_frac=0.012, fs_lab=5, fs_unit=5, unit_text="km"
        )
        if pt is not None:
            cx_data, cy_data = pt
            cx_ax, cy_ax = ax.transAxes.inverted().transform(ax.transData.transform((cx_data, cy_data)))
            Robyn_paper_2_defs.add_north_arrow_axes(
                ax, cx_ax, cy_ax,
                size_frac=0.080, gap_frac=0.050,
                shaft_w_frac=0.10, head_w_frac=0.30, head_h_frac=0.50,
                fs=5, lw=0.5
            )
    # Save
    out_png = out_dir / "avoided_fluvial_ead_300m_linear_USD_plain_figure_3a_min_supplementary.png"
    fig.savefig(out_png, dpi=600, bbox_inches="tight", facecolor="white")
    plt.show()
    print("Saved:", out_png)